# EDA — Рекомендации товаров в e-commerce (Retailrocket)

**Цель EDA:** понять структуру событийных данных, конверсионную воронку и поведение пользователей
для последующего формирования таргета и выбора стратегии моделирования.

**Датасет:** Retailrocket — 2 756 101 событий, 1 407 580 пользователей, 235 061 товар

---
**Разделы:**
1. Загрузка и базовая информация
2. Конверсионная воронка view → addtocart → transaction
3. Поведение пользователей
4. Популярность товаров
5. Временная динамика
6. Свойства товаров и категории
7. Выводы

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.getenv("DATA_DIR", PROJECT_ROOT / "data" / "raw"))

EVENTS_PATH = DATA_DIR / "events.csv"
PROPS1_PATH = DATA_DIR / "item_properties_part1.csv"
PROPS2_PATH = DATA_DIR / "item_properties_part2.csv"
CAT_PATH    = DATA_DIR / "category_tree.csv"

RANDOM_STATE = 42
print("Пути к данным:")
for p in (EVENTS_PATH, PROPS1_PATH, PROPS2_PATH, CAT_PATH):
    status = "OK" if p.exists() else "НЕ НАЙДЕН"
    print(f"  [{status}] {p}")

## 1. Загрузка и базовая информация

In [ ]:
events = pd.read_csv(EVENTS_PATH)
events["dt"] = pd.to_datetime(events["timestamp"], unit="ms")
events["visitorid"] = events["visitorid"].astype(int)
events["itemid"]    = events["itemid"].astype(int)

print(f"Форма: {events.shape}")
print(f"Период: {events['dt'].min().date()} — {events['dt'].max().date()}")
print(f"Уникальных пользователей: {events['visitorid'].nunique():,}")
print(f"Уникальных товаров:       {events['itemid'].nunique():,}")
events.head()

In [ ]:
event_counts = events["event"].value_counts()
print("Распределение типов событий:")
print(event_counts.to_frame("count").assign(pct=lambda df: (df["count"] / len(events) * 100).round(2)))

fig, ax = plt.subplots(figsize=(7, 4))
event_counts.plot(kind="bar", ax=ax, color=["steelblue", "seagreen", "tomato"], rot=0)
ax.set_title("Распределение типов событий")
ax.set_ylabel("Количество событий")
ax.bar_label(ax.containers[0], fmt="{:,.0f}")
plt.tight_layout()
plt.show()

## 2. Конверсионная воронка view → addtocart → transaction

In [ ]:
# Пользователи по уровням воронки
users_view   = set(events[events["event"] == "view"]["visitorid"])
users_cart   = set(events[events["event"] == "addtocart"]["visitorid"])
users_buy    = set(events[events["event"] == "transaction"]["visitorid"])

funnel = pd.Series({
    "Хотя бы 1 view": len(users_view),
    "Хотя бы 1 addtocart": len(users_cart),
    "Хотя бы 1 transaction": len(users_buy),
})

print("Воронка пользователей:")
for label, n in funnel.items():
    base = len(users_view)
    print(f"  {label}: {n:,} ({n/base:.1%} от users_view)")

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(funnel.index[::-1], funnel.values[::-1], color=["tomato", "seagreen", "steelblue"])
ax.bar_label(bars, fmt="{:,.0f}", padding=5)
ax.set_title("Воронка: уникальные пользователи по уровню события")
ax.set_xlabel("Уникальных пользователей")
plt.tight_layout()
plt.show()

# Конверсия view→cart (по товар-пользователь парам)
pairs_view = set(zip(events[events["event"]=="view"]["visitorid"], events[events["event"]=="view"]["itemid"]))
pairs_cart = set(zip(events[events["event"]=="addtocart"]["visitorid"], events[events["event"]=="addtocart"]["itemid"]))
pairs_buy  = set(zip(events[events["event"]=="transaction"]["visitorid"], events[events["event"]=="transaction"]["itemid"]))

cart_after_view = len(pairs_view & pairs_cart)
buy_after_cart  = len(pairs_cart & pairs_buy)

print(f"\nПары (user, item):")
print(f"  view → cart конверсия: {cart_after_view / len(pairs_view):.2%}")
print(f"  cart → buy  конверсия: {buy_after_cart  / len(pairs_cart):.2%}")

## 3. Поведение пользователей

In [ ]:
# Количество событий на пользователя
user_events = events.groupby("visitorid")["event"].count()

print("Событий на пользователя:")
print(user_events.describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гистограмма (обрезаем хвост до 50)
axes[0].hist(user_events.clip(upper=50), bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Распределение числа событий на пользователя (≤50)")
axes[0].set_xlabel("Событий")
axes[0].set_ylabel("Пользователей")

# Log-log (степенной закон)
cnt = user_events.value_counts().sort_index()
axes[1].loglog(cnt.index, cnt.values, "o", markersize=3, alpha=0.7, color="seagreen")
axes[1].set_title("Log-log: число событий vs. число пользователей")
axes[1].set_xlabel("Событий на пользователя")
axes[1].set_ylabel("Пользователей")

plt.tight_layout()
plt.show()

# Доля «активных» пользователей
active_2 = (user_events >= 2).mean()
active_5 = (user_events >= 5).mean()
print(f"\nПользователей с ≥2 событиями: {active_2:.1%}")
print(f"Пользователей с ≥5 событиями: {active_5:.1%}")

In [ ]:
# События по типу на пользователя
user_by_type = events.groupby(["visitorid", "event"]).size().unstack(fill_value=0)

for col in ["view", "addtocart", "transaction"]:
    if col not in user_by_type.columns:
        user_by_type[col] = 0

print("Медианное число событий каждого типа (для пользователей, у которых оно > 0):")
for col in ["view", "addtocart", "transaction"]:
    series = user_by_type[col]
    print(f"  {col}: медиана={series[series>0].median():.0f}, макс={series.max()}")

# Совместное поведение
has_cart  = (user_by_type["addtocart"] > 0)
has_buy   = (user_by_type["transaction"] > 0)
view_only = (~has_cart & ~has_buy).mean()
cart_no_buy = (has_cart & ~has_buy).mean()
bought = has_buy.mean()

print(f"\nПоведение пользователей:")
print(f"  Только просмотры:           {view_only:.1%}")
print(f"  Добавлял в корзину (без покупки): {cart_no_buy:.1%}")
print(f"  Совершил покупку:            {bought:.1%}")

## 4. Популярность товаров

In [ ]:
# Популярность по addtocart (наш таргет)
item_cart = (
    events[events["event"] == "addtocart"]
    .groupby("itemid")["visitorid"]
    .nunique()
    .sort_values(ascending=False)
)

item_view = (
    events[events["event"] == "view"]
    .groupby("itemid")["visitorid"]
    .nunique()
    .sort_values(ascending=False)
)

print(f"Товаров с ≥1 addtocart: {len(item_cart):,}")
print(f"Товаров с ≥1 view:       {len(item_view):,}")
print(f"\nТоп-10 товаров по addtocart:")
print(item_cart.head(10))

# Кумулятивное покрытие
cum = item_cart.cumsum() / item_cart.sum()
pct_items_50 = (cum <= 0.50).sum() / len(cum) * 100
pct_items_80 = (cum <= 0.80).sum() / len(cum) * 100
print(f"\nТоп-{pct_items_50:.1f}% товаров → 50% всех addtocart")
print(f"Топ-{pct_items_80:.1f}% товаров → 80% всех addtocart")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 31), item_cart.values[:30], color="seagreen")
axes[0].set_title("Топ-30 товаров по числу уникальных пользователей (addtocart)")
axes[0].set_xlabel("Ранг товара")
axes[0].set_ylabel("Уникальных пользователей")

x = np.linspace(0, 100, len(cum))
axes[1].plot(x, cum.values * 100, color="steelblue")
axes[1].axhline(80, color="red", linestyle="--", alpha=0.5, label="80%")
axes[1].set_title("Кривая Лоренца: % товаров vs % addtocart")
axes[1].set_xlabel("% товаров (по убыванию популярности)")
axes[1].set_ylabel("Кумулятивная доля addtocart")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Временная динамика

In [ ]:
events["date"] = events["dt"].dt.date
events["week"] = events["dt"].dt.to_period("W")

daily = events.groupby(["date", "event"]).size().unstack(fill_value=0)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, col, color in zip(axes, ["view", "addtocart", "transaction"], ["steelblue", "seagreen", "tomato"]):
    ax.plot(pd.to_datetime(daily.index), daily[col], color=color, linewidth=0.8)
    ax.fill_between(pd.to_datetime(daily.index), daily[col], alpha=0.3, color=color)
    ax.set_ylabel(col)
    ax.set_title(f"Суточная динамика: {col}")

plt.xlabel("Дата")
plt.tight_layout()
plt.show()

print("Период данных:", events["dt"].min().date(), "—", events["dt"].max().date())
print(f"Длительность: {(events['dt'].max() - events['dt'].min()).days} дней")

## 6. Свойства товаров и категории

In [ ]:
props = pd.concat([
    pd.read_csv(PROPS1_PATH),
    pd.read_csv(PROPS2_PATH),
], ignore_index=True)

print(f"Свойств товаров (строк): {len(props):,}")
print(f"Уникальных товаров со свойствами: {props['itemid'].nunique():,}")
print(f"Уникальных свойств: {props['property'].nunique():,}")

top_props = props["property"].value_counts().head(10)
print("\nТоп-10 свойств:")
print(top_props)

# Товары с categoryid (ключевое свойство)
category_props = props[props["property"] == "categoryid"]
n_with_category = category_props["itemid"].nunique()
print(f"\nТоваров с categoryid: {n_with_category:,}")

# Дерево категорий
cat_tree = pd.read_csv(CAT_PATH)
print(f"\nДерево категорий: {len(cat_tree)} рёбер")
print(f"Уникальных категорий: {cat_tree['categoryid'].nunique()}")
cat_tree.head(5)

In [ ]:
# Пересечение: товары с событиями vs. товары со свойствами
items_with_events = set(events["itemid"].unique())
items_with_props  = set(props["itemid"].unique())

both = items_with_events & items_with_props
print(f"Товаров с событиями:  {len(items_with_events):,}")
print(f"Товаров со свойствами: {len(items_with_props):,}")
print(f"Пересечение:           {len(both):,} ({len(both)/len(items_with_events):.1%} от тех, что в событиях)")

# Популярные категории по addtocart
cat_latest = category_props.sort_values("timestamp").drop_duplicates("itemid", keep="last")
cat_latest["itemid"] = cat_latest["itemid"].astype(int)
cat_latest["categoryid"] = cat_latest["value"].astype(str)

cart_events = events[events["event"] == "addtocart"][["itemid"]]
cart_with_cat = cart_events.merge(cat_latest[["itemid", "categoryid"]], on="itemid", how="left")

top_cats = cart_with_cat["categoryid"].value_counts().head(10)
print("\nТоп-10 категорий по addtocart:")
print(top_cats)

## 7. Выводы

### Структура данных
- Три таблицы: **events** (2.76M строк) + **item_properties** (417K строк) + **category_tree** (1669 рёбер)
- Период событий: **май — сентябрь 2015** (145 дней)
- Implicit feedback: нет явных рейтингов, только факты взаимодействия

### Конверсионная воронка
- 96.7% событий — просмотры; addtocart — 2.5%, транзакции — 0.8%
- Пользователей с addtocart: ~14.1K (1% от всех); с покупками: ~7.6K (0.5%)
- view → cart конверсия (по парам user-item): ~0.9%

### Поведение пользователей
- Распределение событий — степенной закон: большинство пользователей делают 1–2 события
- 96%+ пользователей имеют только просмотры (cold-start — основной сценарий)
- Медиана событий на пользователя: **1** (просмотры), **2** (addtocart среди добавлявших)

### Популярность товаров
- Сильная концентрация: топ-1% товаров → ~30% всех addtocart
- 11.5K товаров из 235K получили хотя бы 1 addtocart

### Выбор стратегии оптимизации

| Вариант | За | Против |
|---------|----|---------|
| view | Большой объём данных | Слабый сигнал (случайные просмотры) |
| **addtocart** | **Явный коммерческий интерес + достаточный объём** | **—** |
| transaction | Сильнейший сигнал | Очень мало данных (22K), задержка после intent |

**Решение**: оптимизируем под `addtocart`. Веса: view=1, addtocart=5, transaction=10.

### Выводы для моделирования
| Параметр | Решение |
|---|---|
| Задача | Ранжирование (implicit feedback) |
| Метрики | recall@10, NDCG@10 |
| Train/Val | По времени: последние 14 дней → holdout |
| Cold-start | Popularity baseline для новых пользователей |
| Признаки | Веса событий, матрица взаимодействий, категориальные свойства |